In [116]:
import pandas as pd

Units = pd.read_csv('./Units.csv')
Lines = pd.read_csv('./Lines.csv')
Links = pd.read_csv('./Links.csv')
Transformers = pd.read_csv('./Transformers.csv')
Buses = pd.read_csv('./Buses.csv') 
Loads = pd.read_csv('./Loads.csv')

file_path = "./GB_System.xlsx"

National_Data = pd.read_excel('/Users/zm348/PhD/Projects/Nature-EV/data/Inputs_prepared/GB_System_origin.xlsx', sheet_name='national_data')
Loads

,LoadID,bus_name,load_weight,RegionName
0,1,way/92419253-275,0.002569,Macclesfield
1,2,way/1190395478-220,0.000261,Strichen
2,3,way/262523325-400,0.000241,Dunbar
3,4,way/49499923-400,0.004041,Hams Hall
4,5,way/25935453-400,0.003187,Burwell Main
...,...,...,...,...
380,381,GB71-275,0.001419,Upper Boat
381,382,GB52-400,0.005545,NaN
382,383,way/87464485-400,0.004038,Rye House
383,384,way/87464485-400,0.004038,Rye House


## Initialize

In [ ]:
# with pd.ExcelWriter(file_path, mode='a', engine='openpyxl') as writer:
    # Lines.to_excel(writer, sheet_name='Lines', index=False)
    # Units.to_excel(writer, sheet_name='Units', index=False)
    # Loads.to_excel(writer, sheet_name='Demands', index=False)
    # Buses.to_excel(writer, sheet_name='Buses', index=False)
    # National_Data.to_excel(writer, sheet_name='national_data', index=False)
    # Transformers.to_excel(writer, sheet_name='Transformers', index=False)
    # Links.to_excel(writer, sheet_name='Links', index=False)

In [14]:
GB_System = pd.read_excel(file_path, sheet_name=None)

## Bus ID Mapping

In [2]:
Buses['BusID'] = range(len(Buses))
Busname_to_BusID = dict(zip(Buses['bus_name'], Buses['BusID']))

## line

In [4]:
Lines.rename(columns={'LineID': 'Line_ID'}, inplace=True)
Lines['NodeIn'] = Lines['bus0'].map(Busname_to_BusID)
Lines['NodeOut'] = Lines['bus1'].map(Busname_to_BusID)
Lines.rename(columns={'capacity': 'cap'}, inplace=True)
Lines.rename(columns={'x': 'xl'}, inplace=True)

Links.rename(columns={'LinkID': 'Line_ID'}, inplace=True)
Links['NodeIn'] = Links['bus0'].map(Busname_to_BusID)
Links['NodeOut'] = Links['bus1'].map(Busname_to_BusID)
Links.rename(columns={'capacity': 'cap'}, inplace=True)
Links.rename(columns={'x': 'xl'}, inplace=True)

Transformers.rename(columns={'TransformerID': 'Line_ID'}, inplace=True)
Transformers['NodeIn'] = Transformers['bus0'].map(Busname_to_BusID)
Transformers['NodeOut'] = Transformers['bus1'].map(Busname_to_BusID)
Transformers.rename(columns={'s_nom': 'cap'}, inplace=True)

line = pd.concat([Lines, Links, Transformers], ignore_index=True)
line = line[['Line_ID', 'NodeIn', 'NodeOut', 'cap', 'xl']]
line['Line_ID'] = range(len(line))
line

,Line_ID,NodeIn,NodeOut,cap,xl
0,0,175.0,419.0,921.668,14.577213
1,1,99.0,88.0,3574.953,3.447219
2,2,118.0,35.0,1843.335,0.310518
3,3,474.0,128.0,3574.953,3.492045
4,4,139.0,74.0,1474.668,7.949573
...,...,...,...,...,...
735,735,372.0,NaN,NaN,NaN
736,736,376.0,216.0,NaN,NaN
737,737,386.0,NaN,NaN,NaN
738,738,388.0,315.0,NaN,NaN


In [ ]:
line.loc[line['NodeOut'].isna(), 'NodeOut'] = line.loc[line['NodeOut'].isna(), 'NodeIn']
line['cap'] = line['cap'].fillna(line['cap'].mean())
line['xl'] = line['xl'].fillna(line['xl'].mean())

# p.u.
Xbase = 0.0005
line['xl'] = line['xl']/Xbase

## Loads Mapping

In [ ]:
Loads_merged = Loads.groupby('bus_name', as_index=False)['load_weight'].sum()
BusID_to_Loads = dict(zip(Loads_merged['bus_name'].map(Busname_to_BusID), Loads_merged['load_weight']))

np.float64(0.9999999689999999)

## dem

In [22]:
dem = pd.DataFrame()
dem['Bus_ID'] = Buses['BusID']
dem['No'] = range(1, len(dem)+1)
dem['base'] = dem['Bus_ID'].map(BusID_to_Loads)
dem['base'] = dem['base'].fillna(0)
dem

,Bus_ID,No,base
0,0,1,0.000000
1,1,2,0.000426
2,2,3,0.000000
3,3,4,0.000000
4,4,5,0.000407
...,...,...,...
538,538,539,0.000000
539,539,540,0.000000
540,540,541,0.000000
541,541,542,0.000000


In [ ]:
mask = National_Data.iloc[:, 0] == 'Demand'
demand_dict = National_Data.loc[mask].iloc[0, 1:].to_dict()

for t_col in demand_dict.keys():
    dem[t_col] = dem['base'] * demand_dict[t_col]

np.float64(28680.625267169522)

## dg

In [ ]:
Units = Units[Units['Status'] == 'operating']
Units['Bus_ID'] = Units['Bus name'].map(Busname_to_BusID)


/var/folders/f0/zj71chhd6pvdlx6llbr36gtr0000gp/T/ipykernel_65968/1420855661.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Units['Bus_ID'] = Units['Bus name'].map(Busname_to_BusID)


In [ ]:
Units = Units[['Technology', 'Bus_ID', 'capacity']]
Units = Units.groupby(['Technology', 'Bus_ID'], as_index=False)['capacity'].sum()
dg = Units.sort_values(by=['Bus_ID', 'Technology']).reset_index(drop=True)
dg['NO'] = range(1, len(dg) + 1)
dg.rename(columns={'Technology': 'Type', 'capacity': 'Pmax'}, inplace=True)

,Type
0,biomass
1,gas
2,hydro
3,nuclear
4,offwind
5,oil
6,onwind
7,solar


In [73]:
tech_costs = {
    'biomass': 98,
    'solar': 44,
    'offwind': 65,
    'onwind': 45,
    'nuclear': 92,
    'hydro': 90,
    'coal':100,
    'oil':130,
    'gas':70
}


carbon_intensity = {
    'biomass': 0,    
    'solar': 0,      
    'offwind': 0,    
    'onwind': 0,     
    'nuclear': 0,    
    'hydro': 0,      
    'coal': 820,     
    'oil': 650,      
    'gas': 490       
}

dg['lambdaG'] = dg['Type'].map(tech_costs)
dg['Carbon'] = dg['Type'].map(carbon_intensity)




## res

In [92]:
res = dg.copy()
res['base'] = res.groupby('Type')['Pmax'].transform(lambda x: x / x.sum())
res = res[['Bus_ID','base','Type']]
res.rename(columns={'Type': 'NO'}, inplace=True)
res = res[res['NO'].isin(['onwind', 'offwind', 'solar'])]
res

,Bus_ID,base,NO
0,0,0.000726,offwind
1,0,0.004365,onwind
3,1,0.005675,onwind
4,1,0.000506,solar
5,4,0.011950,onwind
...,...,...,...
503,486,0.000890,solar
505,488,0.014787,onwind
508,489,0.008997,offwind
510,490,0.033169,onwind


In [ ]:
mask_wind = National_Data.iloc[:, 0] == 'Wind'
mask_solar = National_Data.iloc[:, 0] == 'Solar'
demand_dict_wind = National_Data.loc[mask_wind].iloc[0, 1:].to_dict()
demand_dict_solar = National_Data.loc[mask_solar].iloc[0, 1:].to_dict()

for t_col in demand_dict_wind.keys():  # shared keys
    res[t_col] = 0.0
    res.loc[res['NO'].isin(['onwind', 'offwind']), t_col] = \
        res['base'] * demand_dict_wind[t_col] / 2 # Divide by 2 for onwind and offwind
    res.loc[res['NO'] == 'solar', t_col] = \
        res['base'] * demand_dict_solar[t_col]


# total_T5_solar = res.loc[res['NO'] == 'solar', 'T5'].sum()
# print(total_T0_solar)
# total_T0_wind = res.loc[res['NO'] == 'onwind', 'T0'].sum()
# print(total_T0_wind)
res

,Bus_ID,base,NO,T0,T1,T2,T3,T4,T5,T6,...,T38,T39,T40,T41,T42,T43,T44,T45,T46,T47
0,0,0.000726,offwind,5.068942,5.036077,4.990981,4.982934,4.957522,4.929541e+00,4.913328,...,5.327406,5.333906,5.343058,5.338506,5.330380,5.308462,5.264109,5.207561,5.180289,5.114096
1,0,0.004365,onwind,30.495401,30.297681,30.026376,29.977966,29.825080,2.965674e+01,29.559205,...,32.050350,32.089456,32.144516,32.117130,32.068241,31.936380,31.669548,31.329347,31.165279,30.767053
3,1,0.005675,onwind,39.649252,39.392182,39.039440,38.976498,38.777720,3.855885e+01,38.432037,...,41.670953,41.721798,41.793385,41.757778,41.694214,41.522771,41.175844,40.733524,40.520209,40.002446
4,1,0.000506,solar,0.000000,0.000000,0.000000,0.000000,0.000000,4.417508e-10,0.000733,...,0.003275,0.000009,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
5,4,0.011950,onwind,83.483122,82.941851,82.199137,82.066610,81.648076,8.118724e+01,80.920226,...,87.739895,87.846951,87.997681,87.922709,87.788872,87.427894,86.697424,85.766101,85.316957,84.226786
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
503,486,0.000890,solar,0.000000,0.000000,0.000000,0.000000,0.000000,7.774814e-10,0.001290,...,0.005764,0.000016,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
505,488,0.014787,onwind,103.307748,102.637942,101.718857,101.554859,101.036936,1.004667e+02,100.136244,...,108.575371,108.707850,108.894373,108.801598,108.635979,108.189279,107.285346,106.132864,105.577061,104.228008
508,489,0.008997,offwind,62.854886,62.447360,61.888166,61.788386,61.473270,6.112631e+01,60.925267,...,66.059833,66.140436,66.253921,66.197474,66.096708,65.824925,65.274951,64.573753,64.235590,63.414794
510,490,0.033169,onwind,231.723203,230.220801,228.159258,227.791404,226.629685,2.253506e+02,224.609398,...,243.538681,243.835835,244.254214,244.046115,243.674627,242.672662,240.645105,238.060044,236.813358,233.787381


# Write datas into xlsx

In [ ]:
# with pd.ExcelWriter('GB_System.xlsx', mode='a', if_sheet_exists='replace', engine='openpyxl') as writer:
#     line.to_excel(writer, sheet_name='line', index=False)
#     dem.to_excel(writer, sheet_name='dem', index=False)
#     res.to_excel(writer, sheet_name='res', index=False)
#     dg.to_excel(writer, sheet_name='dg', index=False)

